In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split 
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
df = pd.read_csv('https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv')
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


#### 1. Explorar os dados

In [2]:
df.shape # quantas linhas e colunas tem

(768, 9)

In [3]:
df.dtypes # que tipo de dados tem cada coluna

Pregnancies                   int64
Glucose                       int64
BloodPressure                 int64
SkinThickness                 int64
Insulin                       int64
BMI                         float64
DiabetesPedigreeFunction    float64
Age                           int64
Outcome                       int64
dtype: object

In [4]:
df.isnull().sum() # verificar se há valores NaN

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

In [5]:
df['Outcome'].value_counts() # quantas pessoas tem diabetes (1) e quantas não tem (0)

Outcome
0    500
1    268
Name: count, dtype: int64

In [6]:
df.describe()
# conclusão: existem valores 0 (min) em:
#   -- Glucose,
#   -- BloodPressure,
#   -- Insulin,
#   -- SkinThickness
#   -- BMI,
# Estes valores não representam medidas reais, mas sim dados em falta,
# pois é fisiologicamente impossivel um ser vivo ter estes indicadores 
# a zero.

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


#### 2. Pré-processamento

In [7]:
# Definir quais colunas não podem ter zeros
colunas_com_zeros = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

# susbtituir os zeros por NaN
df[colunas_com_zeros] = df[colunas_com_zeros].replace(0, np.nan)

df.isnull().sum()

Pregnancies                   0
Glucose                       5
BloodPressure                35
SkinThickness               227
Insulin                     374
BMI                          11
DiabetesPedigreeFunction      0
Age                           0
Outcome                       0
dtype: int64

In [8]:
# substituir os NaN pela mediana e confirmar se desapareceram
df[colunas_com_zeros] = df[colunas_com_zeros].fillna(df[colunas_com_zeros].median())
df.isnull().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

In [9]:
# verificar o valor mínimo
df.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,121.656250,72.386719,29.108073,140.671875,32.455208,0.471876,33.240885,0.348958
std,3.369578,30.438286,12.096642,8.791221,86.383060,6.875177,0.331329,11.760232,0.476951
min,0.000000,44.000000,24.000000,7.000000,14.000000,18.200000,0.078000,21.000000,0.000000
25%,1.000000,99.750000,64.000000,25.000000,121.500000,27.500000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,29.000000,125.000000,32.300000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


#### 3. Separar os dados para o modelo

In [10]:
X = df.drop('Outcome', axis=1)
y = df['Outcome']

print('Dimensões de X:', X.shape)
print('Dimensões de y:', y.shape)

Dimensões de X: (768, 8)
Dimensões de y: (768,)


In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# random_state --> o train_test_split divide os dados aleatoriamente
# antes de os dividir, se não tiver random_state cada vez que executar 
# o código os conjuntos (treino e teste) seriam diferentes, o que altera
# a precisão do modelo a cada execução

print('Treino:', X_train.shape)
print('Teste:', X_test.shape)

# Sem o random_state os dados alteram sempre quando corre
print("Primeiras linhas do treino:")
print(X_train.head(3)) 

Treino: (614, 8)
Teste: (154, 8)
Primeiras linhas do treino:
     Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
60             2     84.0           72.0           29.0    125.0  32.3   
618            9    112.0           82.0           24.0    125.0  28.2   
346            1    139.0           46.0           19.0     83.0  28.7   

     DiabetesPedigreeFunction  Age  
60                      0.304   21  
618                     1.282   50  
346                     0.654   22  


#### 4. Treinar o modelo

In [12]:
# criar o modelo
modelo = DecisionTreeClassifier(random_state=42)

# treinar o modelo com os dados
modelo.fit(X_train, y_train)

# modelo aplica o que aprendeu a dados novos
y_pred = modelo.predict(X_test)

# ver as primeiras 10 previsões e os valores reai
print('Previsões:', y_pred[:10])
print('Reais:', y_test.values[:10])

Previsões: [0 0 0 0 0 1 0 1 1 1]
Reais: [0 0 0 0 0 0 0 0 0 0]


#### 5. Avaliar o modelo

In [13]:
# avaliar a percentagem de previsões corretas -- accuracy
# accuracy = (previsoes certas / total de previsões) * 100
acc = accuracy_score(y_test, y_pred)
print(f'Accuracy: {acc:.2%}')

# CONFUSION MATRIX ----------------------------
#
#                   Previsto: 0     Previsto: 1
#   Real: 0  ->         TN              FP
#   Real: 1  ->         FN              TP
#
#   TN (True Negative)  -> previu 0, era 0 ✅
#   TP (True Positive)  -> previu 1, era 1 ✅
#   FP (False Positive) -> previu 1, era 0 ❌
#   FN (False Negative) -> previu 0, era 1 ❌
#
#----------------------------------------------

cm = confusion_matrix(y_test, y_pred)
print(cm)
# conclusão: preveu erradamente que 20 pessoas não tinham diabetes (0)
#            preveu erradamente que 23 pessoas tinham diabetes (1)
#            preveu corretamente que 76 pessoas tinham diabetes
#            e que 35 pessoas não tinham

Accuracy: 72.08%
[[76 23]
 [20 35]]


#### Extra - Testar com dados novos

In [23]:
print(df.columns)
colunas = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
            'BMI', 'DiabetesPedigreeFunction', 'Age']

Index(['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
       'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'],
      dtype='object')


In [24]:
# caso 1
pessoa = pd.DataFrame([[2, 120, 70, 25, 100, 28.5, 0.3, 35]], columns=colunas)
previsao = modelo.predict(pessoa)
print('Tem diabetes:', previsao[0])

Tem diabetes: 0


In [25]:
# caso 2
pessoa = pd.DataFrame([[2, 180, 70, 25, 100, 28.5, 0.3, 55]], columns=colunas)
previsao = modelo.predict(pessoa)
print('Tem diabetes:', previsao[0])

Tem diabetes: 0


In [29]:
# caso 3
pessoa = pd.DataFrame([[4, 240, 110, 35, 350, 42, 2.1, 28]], columns=colunas)
previsao = modelo.predict(pessoa)
print('Tem diabetes:', previsao[0])

Tem diabetes: 0


In [28]:
# caso 4
pessoa = pd.DataFrame([[6, 168, 80, 35, 0, 38.5, 0.627, 45]], columns=colunas)
previsao = modelo.predict(pessoa)
print('Tem diabetes:', previsao[0])

# caso 5
pessoa = pd.DataFrame([[1, 140, 90, 40, 0, 45.2, 1.2, 33]], columns=colunas)
previsao = modelo.predict(pessoa)
print('Tem diabetes:', previsao[0])

# caso 6
pessoa = pd.DataFrame([[8, 197, 70, 45, 543, 30.5, 0.158, 53]], columns=colunas)
previsao = modelo.predict(pessoa)
print('Tem diabetes:', previsao[0])

Tem diabetes: 0
Tem diabetes: 0
Tem diabetes: 1


In [18]:
# Criar uma série para visualizar melhor
importancias = pd.Series(modelo.feature_importances_, index=['Gravidezes', 'Glicose', 'Pressão', 'Pele', 'Insulina', 'IMC', 'Pedigree', 'Idade'])
print(importancias.sort_values(ascending=False))

Glicose       0.357487
IMC           0.164377
Idade         0.131524
Pedigree      0.115416
Insulina      0.087381
Pressão       0.079928
Pele          0.035927
Gravidezes    0.027960
dtype: float64
